# Connect Your App

**What you will build:** the last mile — calling your deployed agent API (30) from a real front end, so it's something a user actually touches. Maps to n8n's "Connect Your App" project.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/31_connect_your_app.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

## The shape of it

```
  Browser / app  ──HTTP POST /chat──▶  Your FastAPI service (30)  ──▶  Agent  ──▶  LLM
       ▲                                                                            │
       └──────────────────────  JSON reply (or streamed tokens)  ◀─────────────────┘
```

Your agent is now just an HTTP endpoint. *Any* front end that can make a POST request can use it: a React app, a no-code builder like **Lovable** or **Replit**, a Slack bot, a mobile app. You built the hard part; the front end is a thin client.

## One thing to add first: CORS

A browser on a different domain than your API will be blocked unless the API allows it. Add CORS to the FastAPI app from 30 (loosely for a demo; restrict `allow_origins` to your real front end in production):

In [ ]:
# Add to app.py (from notebook 30), right after `app = FastAPI()`:
cors_snippet = '''
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # for a demo. In production: your front-end's URL only.
    allow_methods=["*"],
    allow_headers=["*"],
)
'''
print(cors_snippet)

## Calling it from a web page

A minimal front end is a few lines of JavaScript. For the **plain** endpoint, `fetch` and read the JSON:

```html
<input id="q"><button onclick="send()">Ask</button><pre id="out"></pre>
<script>
async function send() {
  const text = document.getElementById('q').value;
  const res = await fetch('http://localhost:8000/chat', {
    method: 'POST',
    headers: {'content-type': 'application/json'},
    body: JSON.stringify({ text })
  });
  const data = await res.json();
  document.getElementById('out').textContent = data.reply;
}
</script>
```

For the **streaming** endpoint, read the response body as a stream and append chunks as they arrive — that's what makes the UI feel alive:

```javascript
const res = await fetch('http://localhost:8000/chat/stream', {
  method: 'POST', headers: {'content-type': 'application/json'},
  body: JSON.stringify({ text })
});
const reader = res.body.getReader(), dec = new TextDecoder();
const out = document.getElementById('out');
while (true) {
  const { value, done } = await reader.read();
  if (done) break;
  out.textContent += dec.decode(value).replaceAll('data: ', '');
}
```

## The no-code shortcut

You don't have to hand-write the front end. Tools like **Lovable**, **Replit**, or **v0** generate a working UI from a prompt — tell them "a chat box that POSTs `{text}` to `<your-url>/chat` and shows `reply`" and you have an app. This is the exact bridge the n8n course used: the agent is the backend, a generated UI is the face.

```{note}
Two production must-dos before you share a URL: **an API key / auth** on your endpoint (an open agent endpoint is an open bill), and **rate limiting**. Never ship a public endpoint that calls a paid model without both.
```

**What's next:** in **32** you build a real **capstone project** — combining tools, memory, guardrails, and evals into one thing worth putting in a portfolio.